## 基础环境与路径设置

In [ ]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt
import jax

# 【关键】把包含 'time_opt_erg_lib' 的父目录加入系统路径
# 假设你的 notebook 在 experiments/bias_search 目录下
# 你可能需要向上两级或三级找到 time_opt_erg_lib
# 请根据实际情况调整下面这一行：
sys.path.append('../../..') 
# 或者用绝对路径更保险：
sys.path.append('/home/songxy/code/time-optimal-ergodic-search')

# 验证一下是否能导入核心库
try:
    from time_opt_erg_lib.dynamics import DoubleIntegrator
    print("✅ 成功导入 time_opt_erg_lib")
except ImportError as e:
    print("❌ 导入失败，请检查 sys.path.append 的路径是否正确")
    print(e)

## RandomTargetDistribution (Cleaned / No ROS)

In [ ]:
# ## Cell: RandomTargetDistribution (JAX/Numpy Namespace Fixed)

import jax.numpy as jnp   # 【关键】JAX数学库，用于自动求导和vmap
import numpy as np        # 【关键】标准Numpy，用于随机数生成和初始化
from jax import vmap
import matplotlib.pyplot as plt
import time

# --- 1. 定义父类 (TargetDistribution) ---
class TargetDistribution:
    def __init__(self, wrksp_bnds, grid_res=100):
        self.wrksp_bnds = wrksp_bnds
        self.grid_res = grid_res
        
        # 使用标准 numpy 生成坐标网格 (初始化时不需要梯度)
        x = np.linspace(wrksp_bnds[0][0], wrksp_bnds[0][1], grid_res)
        y = np.linspace(wrksp_bnds[1][0], wrksp_bnds[1][1], grid_res)
        self.X, self.Y = np.meshgrid(x, y)
        
        # 【关键】转为 jnp.stack，因为这个变量后面要喂给 JAX 的 vmap
        # [N*N, 2]
        self.grid_pts = jnp.stack([self.X.flatten(), self.Y.flatten()], axis=1)

    def p(self, x):
        """ 需要子类实现 """
        raise NotImplementedError

    @property
    def evals(self):
        """
        返回 (values, grid_pts) 元组
        values: [N*N] 扁平化 JAX 数组
        """
        # 1. 计算概率值 (使用 vmap 加速)
        # self.p 内部必须全用 jnp 操作
        vals = vmap(self.p)(self.grid_pts)
        
        # 2. 归一化
        vals = vals / jnp.sum(vals)
        
        # 3. 返回元组
        return (vals, self.grid_pts)

    @property
    def domain(self):
        return self.X, self.Y


# --- 2. 您的原始类 (修复了 np/jnp 混用问题) ---
class RandomTargetDistribution(TargetDistribution):
    def __init__(self, wrksp_bnds, n_gaussians=None, centers=None, covs=None, weights=None, seed=None):
        """
        创建随机或指定的高斯混合分布
        """
        # --- 初始化阶段使用标准 np ---
        if seed is not None:
            np.random.seed(seed)
        else:
            np.random.seed(int(time.time()))
        
        x_min, x_max = wrksp_bnds[0]
        y_min, y_max = wrksp_bnds[1]
        x_range = x_max - x_min
        y_range = y_max - y_min
        
        if n_gaussians is None:
            n_gaussians = np.random.randint(1, 8)
        
        if centers is None:
            centers = []
            min_distance = min(x_range, y_range) / 6
            
            for _ in range(n_gaussians):
                max_attempts = 50
                valid_point = False
                for _ in range(max_attempts):
                    candidate = [
                        np.random.uniform(x_min + 0.1*x_range, x_max - 0.1*x_range),
                        np.random.uniform(y_min + 0.1*y_range, y_max - 0.1*y_range)
                    ]
                    if not centers:
                        valid_point = True; break
                    
                    dists = [np.linalg.norm(np.array(candidate) - np.array(c)) for c in centers]
                    if min(dists) >= min_distance:
                        valid_point = True; break
                
                if valid_point: centers.append(candidate)
                else: centers.append(candidate) # fallback
                    
        if covs is None:
            covs = [np.random.uniform(8.0, 30.0) for _ in range(n_gaussians)]
        
        if weights is None:
            w = np.random.uniform(0.5, 1.5, size=n_gaussians)
            weights = w / sum(w)
        
        self.n_gaussians = n_gaussians
        self.centers = centers
        self.covs = covs
        self.weights = weights
        
        # 调用父类初始化
        super(RandomTargetDistribution, self).__init__(wrksp_bnds)
    
    def p(self, x):
        """
        【关键修复】所有数学运算必须使用 jnp (JAX Numpy)
        因为 x 是一个 JAX Tracer 对象
        """
        result = 0.0
        for i in range(self.n_gaussians):
            # 将 list 转为 jnp.array，然后与 tracer 计算
            diff = x[:2] - jnp.array(self.centers[i]) 
            exponent = -self.covs[i] * jnp.sum(diff**2) # 使用 jnp.sum
            result += self.weights[i] * jnp.exp(exponent) # 使用 jnp.exp
        return result
    
    def get_distribution_params(self):
        return {
            "n_gaussians": self.n_gaussians,
            "centers": self.centers,
            "covs": self.covs,
            "weights": self.weights
        }
    
    def visualize(self, title=None, save_path=None):
        fig, ax = plt.subplots(figsize=(10, 8))
        X, Y = self.domain
        # evals返回的是(vals, grid)，取vals并reshape
        # 注意：这里需要把 JAX array 转回 numpy 用于绘图，否则 matplotlib 可能会警告
        Z = np.array(self.evals[0]).reshape(X.shape) 
        
        contour = ax.contourf(X, Y, Z, levels=20, cmap='viridis')
        for i, center in enumerate(self.centers):
            ax.plot(center[0], center[1], 'ro', markersize=10)
            ax.text(center[0], center[1], f"{i+1}", color='white', 
                    fontsize=12, ha='center', va='center')
        
        if title: ax.set_title(title)
        else: ax.set_title(f"Random Distribution with {self.n_gaussians} Gaussians")
        
        plt.colorbar(contour, ax=ax)
        if save_path:
            plt.savefig(save_path, bbox_inches='tight', dpi=300)
            plt.close()
        else:
            plt.tight_layout()
            plt.show()
        return fig, ax

## build_solver_wild.py

In [ ]:
# ## Build Solver Wild (Explicit JAX Fix)
import sys 
sys.path.append('../../..') 

import jax
import jax.numpy as jnp  # 【关键】显式命名为 jnp
from jax import vmap, jit
import numpy as np       # 标准 numpy 命名为 np

from time_opt_erg_lib.dynamics import DoubleIntegrator 
from time_opt_erg_lib.ergodic_metric import ErgodicMetric
from time_opt_erg_lib.fourier_utils import BasisFunc, get_phik, get_ck
from time_opt_erg_lib.opt_solver import AugmentedLagrangeSolver

def build_erg_time_opt_solver(args, target_distr):
    
    workspace_bnds = args['wrksp_bnds']

    # 1. 映射函数 (State -> Workspace [0,1])
    # 【关键修复】使用 jnp.stack 确保调用的是 JAX 操作
    def emap(x):
        val_x = (x[0]-workspace_bnds[0][0])/(workspace_bnds[0][1]-workspace_bnds[0][0])
        val_y = (x[1]-workspace_bnds[1][0])/(workspace_bnds[1][1]-workspace_bnds[1][0])
        return jnp.stack([val_x, val_y])  # <--- jnp.stack
            
    # 2. 初始化核心组件
    basis       = BasisFunc(n_basis=[8,8], emap=emap)
    erg_metric  = ErgodicMetric(basis)
    robot_model = DoubleIntegrator()
    n, m = robot_model.n, robot_model.m

    # 3. 计算目标分布的傅里叶系数
    # get_phik 内部会使用 JAX 操作，传入 JAX 数组是安全的
    args.update({
        'phik' : get_phik(target_distr.evals, basis),
    })

    # 4. 定义边界势场
    def barrier_cost(e):
        return (jnp.maximum(0, e-1) + jnp.maximum(0, -e))**2 # <--- jnp

    # 5. 定义 Loss
    # @jit
    def loss(params, args):
        x = params['x']
        tf = params['tf']
        e = vmap(emap)(x)
        return 100.*jnp.sum(barrier_cost(e)) + tf # <--- jnp

    # 6. 等式约束
    def eq_constr(params, args):
        x = params['x']
        u = params['u']
        x0 = args['x0']
        xf = args['xf']
        tf = params['tf']
        N = args['N']
        dt = tf/N
        return jnp.vstack([  # <--- jnp
            x[0] - x0, 
            x[1:,:]-(x[:-1,:]+dt*vmap(robot_model.dfdt)(x[:-1,:], u[:-1,:])),
            x[-1] - xf
        ])

    # 7. 不等式约束 (无避障)
    def ineq_constr(params, args):
        x = params['x']
        u = params['u']
        phik = args['phik']
        tf = params['tf']
        N = args['N']
        dt = tf/N
        
        ck = get_ck(x, basis, tf, dt)
        val_erg = erg_metric(ck, phik)
        
        _erg_ineq = [10. * jnp.array([val_erg - args['erg_ub'], -tf])] # <--- jnp
        _ctrl_box = [(jnp.abs(u) - 2.).flatten()] # <--- jnp
        
        return jnp.concatenate(_erg_ineq + _ctrl_box) # <--- jnp

    # 8. 初始化求解器
    # 初始猜测可以用标准 numpy 生成，反正传进去会被转
    x = np.linspace(args['x0'], args['xf'], args['N'], endpoint=True)
    u = np.zeros((args['N'], robot_model.m))
    init_sol = {'x': x, 'u' : u, 'tf': np.array(15.0)} 
    
    solver = AugmentedLagrangeSolver(
                    init_sol,
                    loss, 
                    eq_constr, 
                    ineq_constr, 
                    args, 
                    step_size=1e-3,
                    c=1.0)
    
    return solver, []

## data_collection

### 固定自左下到右上

In [ ]:
# data_collection.py
import os
import json
import time
import datetime
import argparse
from tqdm import tqdm
import numpy as np
import jax.numpy as jnp
from jax.dlpack import to_dlpack
import matplotlib.pyplot as plt
# import numpy as onp


class NumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        # 处理JAX数组
        if isinstance(obj, jnp.ndarray):
            # 将JAX数组转换为NumPy数组
            return np.array(obj).tolist()
        # 处理NumPy数组和其他NumPy类型
        elif isinstance(obj, (np.ndarray, np.number)):
            return obj.tolist()
        # 处理其他NumPy类型
        elif isinstance(obj, (np.bool_, np.integer, np.floating, np.complexfloating)):
            return obj.item()
        return super(NumpyEncoder, self).default(obj)
    
class DataCollector:
    def __init__(self, output_dir, workspace_bounds=None,
                 randomize_start_end=False, min_separation=0.5,
                 start_margin=0.05, end_margin=None, max_attempts=1):
        """初始化数据采集器
        参数:
        - output_dir: 输出目录
        - workspace_bounds: 工作空间边界 [[x_min,x_max],[y_min,y_max]]
        - randomize_start_end: 是否随机化起点/终点
        - min_separation: 起终点最小欧氏距离(仅在随机化时生效)
        - start_margin/end_margin: 与边界保持的最小边距(比例, 相对于边界尺寸)
        - max_attempts: 求解失败时的最大重试次数(重新采样起终点)
        """
        self.output_dir = output_dir

        # 创建输出目录
        os.makedirs(output_dir, exist_ok=True)
        os.makedirs(os.path.join(output_dir, "distributions"), exist_ok=True)
        os.makedirs(os.path.join(output_dir, "trajectories"), exist_ok=True)
        os.makedirs(os.path.join(output_dir, "visualizations"), exist_ok=True)

        # 设置默认工作空间边界(如果未指定)
        if workspace_bounds is None:
            self.workspace_bounds = np.array([[0., 3.5], [-1., 3.5]])
        else:
            self.workspace_bounds = np.array(workspace_bounds)

        # 起终点随机化相关配置
        self.randomize_start_end = bool(randomize_start_end)
        self.min_separation = float(min_separation)
        self.start_margin = float(start_margin)
        self.end_margin = float(end_margin) if end_margin is not None else float(start_margin)
        self.max_attempts = int(max_attempts)

        # # 初始化ROS节点
        # try:
        #     rospy.init_node('ergodic_data_collector', anonymous=True)
        # except Exception as e:
        #     # 如果节点已经初始化或其他问题，记录错误但继续执行
        #     print(f"ROS initialization note: {e}")

        # 数据集索引
        self.dataset_index = {
            "created_at": datetime.datetime.now().isoformat(),
            "workspace_bounds": self.workspace_bounds.tolist(),
            "distributions": [],
            "trajectories": []
        }

    def generate_dataset(self, num_distributions=50, trajectories_per_distribution=20, 
                         gamma_range=(0.001, 0.5), visualize=True):
        """
        生成完整数据集
        
        参数:
        - num_distributions: 要生成的分布数量
        - trajectories_per_distribution: 每个分布生成的轨迹数量
        - gamma_range: γ值范围(最小值,最大值)
        - visualize: 是否生成可视化图像
        """
        print(f"Generating dataset with {num_distributions} distributions and {trajectories_per_distribution} trajectories per distribution")
        
        # 生成γ值列表 (对数均匀分布)
        gamma_values = np.exp(np.linspace(np.log(gamma_range[0]), np.log(gamma_range[1]), trajectories_per_distribution))
        
        # 生成分布
        for dist_idx in tqdm(range(num_distributions), desc="Generating distributions"):
            try:
                # 生成并处理每个分布
                self._process_distribution(dist_idx, gamma_values, visualize)
            except Exception as e:
                print(f"Error processing distribution {dist_idx}: {e}")
        
        # 保存数据集索引
        with open(os.path.join(self.output_dir, "dataset_index.json"), "w") as f:
            json.dump(self.dataset_index, f, indent=2, cls=NumpyEncoder)
        
        print(f"Dataset generation complete.")
        print(f"Generated {len(self.dataset_index['distributions'])} distributions")
        print(f"Generated {len(self.dataset_index['trajectories'])} trajectories")
        
        return self.dataset_index
    
    def _process_distribution(self, dist_idx, gamma_values, visualize):
        try:
            """处理单个分布的数据生成"""
            # 为每个分布生成随机种子
            dist_seed = int(time.time() * 1000) % 10000 + dist_idx
            
            # 创建随机分布
            distribution = RandomTargetDistribution(self.workspace_bounds, seed=dist_seed)
            dist_params = distribution.get_distribution_params()
            
            # 保存分布参数
            dist_id = f"dist_{dist_idx:04d}"
            dist_file = os.path.join(self.output_dir, "distributions", f"{dist_id}.json")
            with open(dist_file, "w") as f:
                json.dump({
                    "id": dist_id,
                    "params": dist_params,
                    "workspace_bounds": self.workspace_bounds
                }, f, indent=2, cls=NumpyEncoder)
            
            self.dataset_index["distributions"].append(dist_id)
            
            # 可视化分布
            if visualize:
                viz_path = os.path.join(self.output_dir, "visualizations", f"{dist_id}.png")
                distribution.visualize(title=f"Distribution {dist_id}", save_path=viz_path)
            
            # 为每个分布生成多个轨迹(不同γ值)
            successful_trajectories = 0
            for traj_idx, gamma in enumerate(gamma_values):
                if self._generate_trajectory(distribution, dist_id, traj_idx, gamma, visualize):
                    successful_trajectories += 1
            
            print(f"Generated {successful_trajectories}/{len(gamma_values)} trajectories for distribution {dist_id}")
        except Exception as e:
            import traceback
            print(f"Error processing distribution {dist_idx}: {e}")
            print(traceback.format_exc())
            return False
    
    def _generate_trajectory(self, distribution, dist_id, traj_idx, gamma, visualize):
        """生成单条轨迹"""
        try:
            # 采样起点与终点（可随机化）
            if self.randomize_start_end:
                xb, yb = self.workspace_bounds[0], self.workspace_bounds[1]
                dx, dy = xb[1] - xb[0], yb[1] - yb[0]
                sx_min, sx_max = xb[0] + self.start_margin * dx, xb[1] - self.start_margin * dx
                sy_min, sy_max = yb[0] + self.start_margin * dy, yb[1] - self.start_margin * dy
                ex_min, ex_max = xb[0] + self.end_margin * dx, xb[1] - self.end_margin * dx
                ey_min, ey_max = yb[0] + self.end_margin * dy, yb[1] - self.end_margin * dy
                # 简单重试以确保最小间距
                for _ in range(100):
                    start_pt = np.array([
                        np.random.uniform(sx_min, sx_max),
                        np.random.uniform(sy_min, sy_max)
                    ])
                    end_pt = np.array([
                        np.random.uniform(ex_min, ex_max),
                        np.random.uniform(ey_min, ey_max)
                    ])
                    if np.linalg.norm(end_pt - start_pt) >= self.min_separation:
                        break
                else:
                    # 退化情况：放宽约束
                    start_pt = np.array([sx_min, sy_min])
                    end_pt = np.array([ex_max, ey_max])
            else:
                start_pt = np.array([0.1, 0.1])
                end_pt = np.array([3.0, 3.0])

            x0 = np.array([start_pt[0], start_pt[1], 0., 0.])
            xf = np.array([end_pt[0], end_pt[1], 0., 0.])

            # 设置参数
            args = {
                'N': 100,
                'x0': x0,
                'xf': xf,
                'erg_ub': gamma,  # 使用gamma作为尔格迪克度量上限
                'alpha': 0.8,
                'wrksp_bnds': self.workspace_bounds
            }
            
            # 创建求解器
            solver, obstacles = build_erg_time_opt_solver(args, distribution)
            
            # 求解优化问题
            t_start = time.time()
            # 修改这里 - 使用正确的参数名称
            sol = solver.solve(max_iter=1000, eps=1e-5)
            
            # 检查解是否存在
            if sol is None:
                # 尝试从solver对象获取解
                if hasattr(solver, 'solution') and solver.solution is not None:
                    sol = solver.solution
                else:
                    print(f"Solver did not return a solution for dist={dist_id}, gamma={gamma}")
                    return False
            
            solve_time = time.time() - t_start
            
            # 获取结果
            # 如果sol是一个字典，直接访问其键
            if isinstance(sol, dict):
                states = sol['x']
                controls = sol['u']
                tf = float(sol['tf'])
            # 否则，尝试从solver对象获取解的组成部分
            else:
                # 假设solver.solution包含必要的键
                if hasattr(solver, 'solution') and solver.solution is not None:
                    states = solver.solution['x']
                    controls = solver.solution['u']
                    tf = float(solver.solution['tf']) if 'tf' in solver.solution else None
                else:
                    print(f"Cannot access solution data for dist={dist_id}, gamma={gamma}")
                    return False
            
            # 如果tf未定义，尝试从其他地方获取
            if tf is None and hasattr(solver, 'tf'):
                tf = float(solver.tf)
            elif tf is None:
                # 使用默认值或从参数计算
                print(f"Warning: Using default time for dist={dist_id}, gamma={gamma}")
                tf = 10.0  # 默认值，可以根据需要调整
            
            dt = tf / args['N']
            
            # 计算尔格迪克度量
            from time_opt_erg_lib.fourier_utils import BasisFunc, get_phik, get_ck
            from time_opt_erg_lib.ergodic_metric import ErgodicMetric
            
            basis = BasisFunc(n_basis=[8,8], emap=lambda x: jnp.array([
    (x[0]-self.workspace_bounds[0][0])/(self.workspace_bounds[0][1]-self.workspace_bounds[0][0]), 
    (x[1]-self.workspace_bounds[1][0])/(self.workspace_bounds[1][1]-self.workspace_bounds[1][0])]))
            phik = get_phik(distribution.evals, basis)
            ck = get_ck(states, basis, tf, dt)
            erg_metric = ErgodicMetric(basis)(ck, phik)
            
            # 构造轨迹结果（包含显式起点/终点）
            start_point = [float(start_pt[0]), float(start_pt[1])]
            end_point = [float(end_pt[0]), float(end_pt[1])]
            trajectory = {
                "states": states,
                "controls": controls,
                "time_step": dt,
                "total_time": tf,
                "ergodic_metric": float(erg_metric),
                "start_point": start_point,
                "end_point": end_point,
            }

            # 为轨迹创建唯一ID
            traj_id = f"{dist_id}_traj_{traj_idx:04d}_g{gamma:.6f}"

            # 确保目录存在
            os.makedirs(os.path.join(self.output_dir, "trajectories"), exist_ok=True)

            # 保存轨迹数据（向后兼容：下游读取仍可从states[0], states[-1]推断）
            traj_file = os.path.join(self.output_dir, "trajectories", f"{traj_id}.json")
            with open(traj_file, "w") as f:
                json.dump({
                    "id": traj_id,
                    "distribution_id": dist_id,
                    "gamma": float(gamma),
                    "states": states,
                    "controls": controls,
                    "time_step": float(dt),
                    "total_time": float(tf),
                    "ergodic_metric": float(erg_metric),
                    "solve_time": float(solve_time),
                    "start_point": start_point,
                    "end_point": end_point,
                }, f, indent=2, cls=NumpyEncoder)

            # 添加到索引
            self.dataset_index["trajectories"].append(traj_id)

            # 可视化轨迹
            if visualize:
                self._visualize_trajectory(distribution, trajectory, traj_id)

            print(f"Successfully generated trajectory {traj_id} with ergodic metric: {erg_metric}")
            return True
            
        except Exception as e:
            import traceback
            print(f"Error solving trajectory for dist={dist_id}, gamma={gamma}: {e}")
            print(traceback.format_exc())
            return False
    
    def _visualize_trajectory(self, distribution, trajectory, traj_id):
        """可视化轨迹"""
        try:
            # 创建图形
            fig, ax = plt.subplots(figsize=(10, 8))
            
            # 绘制分布
            X, Y = distribution.domain
            Z = distribution.evals[0].reshape(X.shape)
            contour = ax.contourf(X, Y, Z, levels=20, cmap='viridis', alpha=0.7)
            
            # 绘制轨迹
            states = np.array(trajectory["states"])
            ax.plot(states[:, 0], states[:, 1], 'r-', linewidth=2)
            
            # 标记起点和终点（优先使用显式字段）
            sp = trajectory.get("start_point")
            ep = trajectory.get("end_point")
            if sp is None:
                sp = states[0, :2]
            if ep is None:
                ep = states[-1, :2]
            ax.plot(sp[0], sp[1], 'go', markersize=8, label='Start')
            ax.plot(ep[0], ep[1], 'bo', markersize=8, label='End')

            # 添加信息
            gamma = float(trajectory["ergodic_metric"])
            total_time = float(trajectory["total_time"])
            ax.set_title(f"Trajectory {traj_id}\nErgodic Metric: {gamma:.6f}, Time: {total_time:.2f}s")

            # 保存图像
            viz_path = os.path.join(self.output_dir, "visualizations", f"{traj_id}.png")
            plt.savefig(viz_path, bbox_inches='tight', dpi=150)
            plt.close(fig)
        except Exception as e:
            print(f"Error visualizing trajectory {traj_id}: {e}")

# ## Cell: Run Data Collection (Main)

# 1. 定义参数 (替代 argparse)
OUTPUT_DIR = "data/ergodic_dataset_no_obs"   # 数据保存地址
NUM_DISTS = 10                 # 分布数量 (总量)
TRAJS_PER_DIST = 5             # 每个分布生成的轨迹数
VISUALIZE = False                # 是否生成可视化图 (跑大量数据建议 False)
GAMMA_RANGE = (0.005, 0.05)      # 【关键】低 Gamma 逼出 S 型轨迹

# 2. 实例化采集器
# 注意：这里 workspace_bounds 使用默认值 [[0, 3.5], [-1, 3.5]]
collector = DataCollector(output_dir=OUTPUT_DIR)

print(f"🚀 开始采集数据...")
print(f"📂 保存路径: {os.path.abspath(OUTPUT_DIR)}")
print(f"📊 计划总量: {NUM_DISTS} 个分布 x {TRAJS_PER_DIST} 条轨迹 = {NUM_DISTS * TRAJS_PER_DIST} 条数据")

# 3. 开始运行
# 注意：这里调用的是我们修改过的 generate_dataset，确保它传入了 gamma_range
collector.generate_dataset(
    num_distributions=NUM_DISTS,
    trajectories_per_distribution=TRAJS_PER_DIST,
    gamma_range=GAMMA_RANGE, 
    visualize=VISUALIZE
)

print("✅ 数据采集完成！")

### ergodic_dataset_wild_full-随机起终点

In [ ]:
# data_collection.py
import os
import json
import time
import datetime
import argparse
from tqdm import tqdm
import numpy as np
import jax.numpy as jnp
from jax.dlpack import to_dlpack
import matplotlib.pyplot as plt
# import numpy as onp


class NumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        # 处理JAX数组
        if isinstance(obj, jnp.ndarray):
            # 将JAX数组转换为NumPy数组
            return np.array(obj).tolist()
        # 处理NumPy数组和其他NumPy类型
        elif isinstance(obj, (np.ndarray, np.number)):
            return obj.tolist()
        # 处理其他NumPy类型
        elif isinstance(obj, (np.bool_, np.integer, np.floating, np.complexfloating)):
            return obj.item()
        return super(NumpyEncoder, self).default(obj)
    
class DataCollector:
    def __init__(self, output_dir, workspace_bounds=None,
                 randomize_start_end=False, min_separation=0.5,
                 start_margin=0.05, end_margin=None, max_attempts=1):
        """初始化数据采集器
        参数:
        - output_dir: 输出目录
        - workspace_bounds: 工作空间边界 [[x_min,x_max],[y_min,y_max]]
        - randomize_start_end: 是否随机化起点/终点
        - min_separation: 起终点最小欧氏距离(仅在随机化时生效)
        - start_margin/end_margin: 与边界保持的最小边距(比例, 相对于边界尺寸)
        - max_attempts: 求解失败时的最大重试次数(重新采样起终点)
        """
        self.output_dir = output_dir

        # 创建输出目录
        os.makedirs(output_dir, exist_ok=True)
        os.makedirs(os.path.join(output_dir, "distributions"), exist_ok=True)
        os.makedirs(os.path.join(output_dir, "trajectories"), exist_ok=True)
        os.makedirs(os.path.join(output_dir, "visualizations"), exist_ok=True)

        # 设置默认工作空间边界(如果未指定)
        if workspace_bounds is None:
            self.workspace_bounds = np.array([[0., 3.5], [-1., 3.5]])
        else:
            self.workspace_bounds = np.array(workspace_bounds)

        # 起终点随机化相关配置
        self.randomize_start_end = bool(randomize_start_end)
        self.min_separation = float(min_separation)
        self.start_margin = float(start_margin)
        self.end_margin = float(end_margin) if end_margin is not None else float(start_margin)
        self.max_attempts = int(max_attempts)

        # # 初始化ROS节点
        # try:
        #     rospy.init_node('ergodic_data_collector', anonymous=True)
        # except Exception as e:
        #     # 如果节点已经初始化或其他问题，记录错误但继续执行
        #     print(f"ROS initialization note: {e}")

        # 数据集索引
        self.dataset_index = {
            "created_at": datetime.datetime.now().isoformat(),
            "workspace_bounds": self.workspace_bounds.tolist(),
            "distributions": [],
            "trajectories": []
        }

    def generate_dataset(self, num_distributions=50, trajectories_per_distribution=20, 
                         gamma_range=(0.001, 0.5), visualize=True):
        """
        生成完整数据集
        
        参数:
        - num_distributions: 要生成的分布数量
        - trajectories_per_distribution: 每个分布生成的轨迹数量
        - gamma_range: γ值范围(最小值,最大值)
        - visualize: 是否生成可视化图像
        """
        print(f"Generating dataset with {num_distributions} distributions and {trajectories_per_distribution} trajectories per distribution")
        
        # 生成γ值列表 (对数均匀分布)
        gamma_values = np.exp(np.linspace(np.log(gamma_range[0]), np.log(gamma_range[1]), trajectories_per_distribution))
        
        # 生成分布
        for dist_idx in tqdm(range(num_distributions), desc="Generating distributions"):
            try:
                # 生成并处理每个分布
                self._process_distribution(dist_idx, gamma_values, visualize)
            except Exception as e:
                print(f"Error processing distribution {dist_idx}: {e}")
        
        # 保存数据集索引
        with open(os.path.join(self.output_dir, "dataset_index.json"), "w") as f:
            json.dump(self.dataset_index, f, indent=2, cls=NumpyEncoder)
        
        print(f"Dataset generation complete.")
        print(f"Generated {len(self.dataset_index['distributions'])} distributions")
        print(f"Generated {len(self.dataset_index['trajectories'])} trajectories")
        
        return self.dataset_index
    
    def _process_distribution(self, dist_idx, gamma_values, visualize):
        try:
            """处理单个分布的数据生成"""
            # 为每个分布生成随机种子
            dist_seed = int(time.time() * 1000) % 10000 + dist_idx
            
            # 创建随机分布
            distribution = RandomTargetDistribution(self.workspace_bounds, seed=dist_seed)
            dist_params = distribution.get_distribution_params()
            
            # 保存分布参数
            dist_id = f"dist_{dist_idx:04d}"
            dist_file = os.path.join(self.output_dir, "distributions", f"{dist_id}.json")
            with open(dist_file, "w") as f:
                json.dump({
                    "id": dist_id,
                    "params": dist_params,
                    "workspace_bounds": self.workspace_bounds
                }, f, indent=2, cls=NumpyEncoder)
            
            self.dataset_index["distributions"].append(dist_id)
            
            # 可视化分布
            if visualize:
                viz_path = os.path.join(self.output_dir, "visualizations", f"{dist_id}.png")
                distribution.visualize(title=f"Distribution {dist_id}", save_path=viz_path)
            
            # 为每个分布生成多个轨迹(不同γ值)
            successful_trajectories = 0
            for traj_idx, gamma in enumerate(gamma_values):
                if self._generate_trajectory(distribution, dist_id, traj_idx, gamma, visualize):
                    successful_trajectories += 1
            
            print(f"Generated {successful_trajectories}/{len(gamma_values)} trajectories for distribution {dist_id}")
        except Exception as e:
            import traceback
            print(f"Error processing distribution {dist_idx}: {e}")
            print(traceback.format_exc())
            return False
    
    def _generate_trajectory(self, distribution, dist_id, traj_idx, gamma, visualize):
        """生成单条轨迹"""
        try:
            # 采样起点与终点（可随机化）
            if self.randomize_start_end:
                xb, yb = self.workspace_bounds[0], self.workspace_bounds[1]
                dx, dy = xb[1] - xb[0], yb[1] - yb[0]
                sx_min, sx_max = xb[0] + self.start_margin * dx, xb[1] - self.start_margin * dx
                sy_min, sy_max = yb[0] + self.start_margin * dy, yb[1] - self.start_margin * dy
                ex_min, ex_max = xb[0] + self.end_margin * dx, xb[1] - self.end_margin * dx
                ey_min, ey_max = yb[0] + self.end_margin * dy, yb[1] - self.end_margin * dy
                # 简单重试以确保最小间距
                for _ in range(100):
                    start_pt = np.array([
                        np.random.uniform(sx_min, sx_max),
                        np.random.uniform(sy_min, sy_max)
                    ])
                    end_pt = np.array([
                        np.random.uniform(ex_min, ex_max),
                        np.random.uniform(ey_min, ey_max)
                    ])
                    if np.linalg.norm(end_pt - start_pt) >= self.min_separation:
                        break
                else:
                    # 退化情况：放宽约束
                    start_pt = np.array([sx_min, sy_min])
                    end_pt = np.array([ex_max, ey_max])
            else:
                start_pt = np.array([0.1, 0.1])
                end_pt = np.array([3.0, 3.0])

            x0 = np.array([start_pt[0], start_pt[1], 0., 0.])
            xf = np.array([end_pt[0], end_pt[1], 0., 0.])

            # 设置参数
            args = {
                'N': 100,
                'x0': x0,
                'xf': xf,
                'erg_ub': gamma,  # 使用gamma作为尔格迪克度量上限
                'alpha': 0.8,
                'wrksp_bnds': self.workspace_bounds
            }
            
            # 创建求解器
            solver, obstacles = build_erg_time_opt_solver(args, distribution)
            
            # 求解优化问题
            t_start = time.time()
            # 修改这里 - 使用正确的参数名称
            sol = solver.solve(max_iter=1000, eps=1e-5)
            
            # 检查解是否存在
            if sol is None:
                # 尝试从solver对象获取解
                if hasattr(solver, 'solution') and solver.solution is not None:
                    sol = solver.solution
                else:
                    print(f"Solver did not return a solution for dist={dist_id}, gamma={gamma}")
                    return False
            
            solve_time = time.time() - t_start
            
            # 获取结果
            # 如果sol是一个字典，直接访问其键
            if isinstance(sol, dict):
                states = sol['x']
                controls = sol['u']
                tf = float(sol['tf'])
            # 否则，尝试从solver对象获取解的组成部分
            else:
                # 假设solver.solution包含必要的键
                if hasattr(solver, 'solution') and solver.solution is not None:
                    states = solver.solution['x']
                    controls = solver.solution['u']
                    tf = float(solver.solution['tf']) if 'tf' in solver.solution else None
                else:
                    print(f"Cannot access solution data for dist={dist_id}, gamma={gamma}")
                    return False
            
            # 如果tf未定义，尝试从其他地方获取
            if tf is None and hasattr(solver, 'tf'):
                tf = float(solver.tf)
            elif tf is None:
                # 使用默认值或从参数计算
                print(f"Warning: Using default time for dist={dist_id}, gamma={gamma}")
                tf = 10.0  # 默认值，可以根据需要调整
            
            dt = tf / args['N']
            
            # 计算尔格迪克度量
            from time_opt_erg_lib.fourier_utils import BasisFunc, get_phik, get_ck
            from time_opt_erg_lib.ergodic_metric import ErgodicMetric
            
            basis = BasisFunc(n_basis=[8,8], emap=lambda x: jnp.array([
    (x[0]-self.workspace_bounds[0][0])/(self.workspace_bounds[0][1]-self.workspace_bounds[0][0]), 
    (x[1]-self.workspace_bounds[1][0])/(self.workspace_bounds[1][1]-self.workspace_bounds[1][0])]))
            phik = get_phik(distribution.evals, basis)
            ck = get_ck(states, basis, tf, dt)
            erg_metric = ErgodicMetric(basis)(ck, phik)
            
            # 构造轨迹结果（包含显式起点/终点）
            start_point = [float(start_pt[0]), float(start_pt[1])]
            end_point = [float(end_pt[0]), float(end_pt[1])]
            trajectory = {
                "states": states,
                "controls": controls,
                "time_step": dt,
                "total_time": tf,
                "ergodic_metric": float(erg_metric),
                "start_point": start_point,
                "end_point": end_point,
            }

            # 为轨迹创建唯一ID
            traj_id = f"{dist_id}_traj_{traj_idx:04d}_g{gamma:.6f}"

            # 确保目录存在
            os.makedirs(os.path.join(self.output_dir, "trajectories"), exist_ok=True)

            # 保存轨迹数据（向后兼容：下游读取仍可从states[0], states[-1]推断）
            traj_file = os.path.join(self.output_dir, "trajectories", f"{traj_id}.json")
            with open(traj_file, "w") as f:
                json.dump({
                    "id": traj_id,
                    "distribution_id": dist_id,
                    "gamma": float(gamma),
                    "states": states,
                    "controls": controls,
                    "time_step": float(dt),
                    "total_time": float(tf),
                    "ergodic_metric": float(erg_metric),
                    "solve_time": float(solve_time),
                    "start_point": start_point,
                    "end_point": end_point,
                }, f, indent=2, cls=NumpyEncoder)

            # 添加到索引
            self.dataset_index["trajectories"].append(traj_id)

            # 可视化轨迹
            if visualize:
                self._visualize_trajectory(distribution, trajectory, traj_id)

            print(f"Successfully generated trajectory {traj_id} with ergodic metric: {erg_metric}")
            return True
            
        except Exception as e:
            import traceback
            print(f"Error solving trajectory for dist={dist_id}, gamma={gamma}: {e}")
            print(traceback.format_exc())
            return False
    
    def _visualize_trajectory(self, distribution, trajectory, traj_id):
        """可视化轨迹"""
        try:
            # 创建图形
            fig, ax = plt.subplots(figsize=(10, 8))
            
            # 绘制分布
            X, Y = distribution.domain
            Z = distribution.evals[0].reshape(X.shape)
            contour = ax.contourf(X, Y, Z, levels=20, cmap='viridis', alpha=0.7)
            
            # 绘制轨迹
            states = np.array(trajectory["states"])
            ax.plot(states[:, 0], states[:, 1], 'r-', linewidth=2)
            
            # 标记起点和终点（优先使用显式字段）
            sp = trajectory.get("start_point")
            ep = trajectory.get("end_point")
            if sp is None:
                sp = states[0, :2]
            if ep is None:
                ep = states[-1, :2]
            ax.plot(sp[0], sp[1], 'go', markersize=8, label='Start')
            ax.plot(ep[0], ep[1], 'bo', markersize=8, label='End')

            # 添加信息
            gamma = float(trajectory["ergodic_metric"])
            total_time = float(trajectory["total_time"])
            ax.set_title(f"Trajectory {traj_id}\nErgodic Metric: {gamma:.6f}, Time: {total_time:.2f}s")

            # 保存图像
            viz_path = os.path.join(self.output_dir, "visualizations", f"{traj_id}.png")
            plt.savefig(viz_path, bbox_inches='tight', dpi=150)
            plt.close(fig)
        except Exception as e:
            print(f"Error visualizing trajectory {traj_id}: {e}")

# ## Cell: Run Data Collection (Main)

# ## Cell: Run Data Collection (Main) with Randomization

# 1. 配置参数
OUTPUT_DIR = "data/ergodic_dataset_wild_full" # 换个名字，别覆盖之前的
NUM_DISTS = 500         # 500 个分布
TRAJS_PER_DIST = 40     # 每个分布 40 条
VISUALIZE = False       # 跑大量数据关闭可视化
GAMMA_RANGE = (0.005, 0.05) 

# 2. 实例化采集器 - 【关键修改】开启随机化
collector = DataCollector(
    output_dir=OUTPUT_DIR,
    # 开启起终点随机化
    randomize_start_end=True, 
    # 最小间隔 1.5m，防止起点终点太近，那样生成的轨迹没意义
    min_separation=1.5,       
    # 留一点边距，别生成在地图边缘
    start_margin=0.1          
)

print(f"🚀 开始采集全量数据 (随机起终点)...")
print(f"📊 总量: {NUM_DISTS * TRAJS_PER_DIST} 条")

# 3. 运行
collector.generate_dataset(
    num_distributions=NUM_DISTS,
    trajectories_per_distribution=TRAJS_PER_DIST,
    gamma_range=GAMMA_RANGE, 
    visualize=VISUALIZE
)

### 固定起终点-自左上至右下

In [ ]:
# data_collection_diagonal_fixed.py
import os
import json
import time
import datetime
from tqdm import tqdm
import numpy as np
import jax.numpy as jnp
import matplotlib
matplotlib.use('Agg') 
import matplotlib.pyplot as plt

# 引入之前的定义 (确保在同一目录下或路径正确)
# 假设 RandomTargetDistribution 和 build_erg_time_opt_solver 已经定义好了
# 这里为了代码独立性，我假设你已经在上面定义了它们，或者从文件导入
# 如果是 Notebook，请确保前面的 Cell 已经运行

# ... (RandomTargetDistribution 类定义保持不变) ...

# 重新定义一下 Encoder 辅助类以防万一
class NumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, jnp.ndarray): return np.array(obj).tolist()
        elif isinstance(obj, (np.ndarray, np.number)): return obj.tolist()
        return super(NumpyEncoder, self).default(obj)
    
class DataCollector:
    def __init__(self, output_dir, workspace_bounds=None):
        self.output_dir = output_dir
        os.makedirs(output_dir, exist_ok=True)
        os.makedirs(os.path.join(output_dir, "distributions"), exist_ok=True)
        os.makedirs(os.path.join(output_dir, "trajectories"), exist_ok=True)
        os.makedirs(os.path.join(output_dir, "visualizations"), exist_ok=True)

        if workspace_bounds is None: self.workspace_bounds = np.array([[0., 3.5], [-1., 3.5]])
        else: self.workspace_bounds = np.array(workspace_bounds)

        self.dataset_index = {
            "created_at": datetime.datetime.now().isoformat(),
            "distributions": [], "trajectories": []
        }

    def generate_dataset(self, num_distributions=10, trajectories_per_distribution=5, 
                         # 【修改1】放宽 Gamma 范围，从 0.03 起步，给 Solver 喘息机会
                         gamma_range=(0.03, 0.15), 
                         visualize=True):
        print(f"🚀 开始采集固定对角线数据: {num_distributions} Dists x {trajectories_per_distribution} Trajs")
        print(f"   Gamma Range: {gamma_range}")
        
        gamma_values = np.exp(np.linspace(np.log(gamma_range[0]), np.log(gamma_range[1]), trajectories_per_distribution))
        
        success_count = 0
        total_attempts = num_distributions * trajectories_per_distribution
        
        for dist_idx in tqdm(range(num_distributions), desc="Generating"):
            # 1. 生成分布
            dist_seed = int(time.time() * 1000) % 10000 + dist_idx
            distribution = RandomTargetDistribution(self.workspace_bounds, seed=dist_seed)
            
            # 保存分布
            dist_id = f"dist_{dist_idx:04d}"
            with open(os.path.join(self.output_dir, "distributions", f"{dist_id}.json"), "w") as f:
                json.dump({"id": dist_id, "params": distribution.get_distribution_params()}, f, indent=2, cls=NumpyEncoder)
            self.dataset_index["distributions"].append(dist_id)

            if visualize:
                distribution.visualize(save_path=os.path.join(self.output_dir, "visualizations", f"{dist_id}_map.png"))

            # 2. 生成轨迹
            for traj_idx, gamma in enumerate(gamma_values):
                if self._generate_diagonal_trajectory(distribution, dist_id, traj_idx, gamma, visualize):
                    success_count += 1
        
        print(f"\n📊 采集报告: 尝试 {total_attempts} 次，成功保存 {success_count} 条。")
        if success_count > 0:
            print("✅ 成功！请检查 visualizations 文件夹。")
        else:
            print("❌ 依然失败。请检查 Solver 内部逻辑。")
        return self.dataset_index
    
    def _generate_diagonal_trajectory(self, distribution, dist_id, traj_idx, gamma, visualize):
        try:
            xb, yb = self.workspace_bounds[0], self.workspace_bounds[1]
            
            # 固定对角线
            start_pt = np.array([xb[0] + 0.2, yb[1] - 0.2]) 
            end_pt   = np.array([xb[1] - 0.2, yb[0] + 0.2])
            x0 = np.array([start_pt[0], start_pt[1], 0., 0.])
            xf = np.array([end_pt[0], end_pt[1], 0., 0.])

            # 【修改2】给予更宽松的时间预算
            # 对角线长约 5.0，之前的系数 4.0 可能略紧，改为 5.0
            dist_val = np.linalg.norm(end_pt - start_pt) 
            tf_estimate = dist_val * 5.0 

            args = {
                'N': 100,      
                'x0': x0,
                'xf': xf,
                'erg_ub': gamma, 
                'alpha': 0.8,
                'wrksp_bnds': self.workspace_bounds
            }
            
            solver, _ = build_erg_time_opt_solver(args, distribution)
            
            # 【修改3】核心修改：放宽 eps，增加 max_iter
            # 即使 tol=0.05 也是可以接受的，所以 eps 设为 1e-2 足够了
            sol = solver.solve(max_iter=3000, eps=1e-2)
            
            # 【修改4】宽容策略：只要 Solver 返回了结果（哪怕标记为 unsuccessful），我们也拿来看看
            # AugmentedLagrangeSolver 通常会返回它找到的最好的解
            states = None
            if isinstance(sol, dict):
                states = sol.get('x') # 即使 fail 也可能有 x
                controls = sol.get('u')
                tf = float(sol.get('tf', tf_estimate))
            elif hasattr(solver, 'solution') and solver.solution: # 某些版本的实现
                states = solver.solution.get('x')
                controls = solver.solution.get('u')
                tf = float(solver.solution.get('tf', tf_estimate))
            
            # 只有当 states 真是 None 或者全是 NaN 时才放弃
            if states is None or np.isnan(states).any():
                # print(f"  [Fail] No solution returned for g={gamma:.3f}")
                return False
            
            # 计算 Metric 验证质量
            dt = tf / args['N']
            # 这里需要重新 import 必要的计算函数
            try:
                from time_opt_erg_lib.fourier_utils import BasisFunc, get_phik, get_ck
                from time_opt_erg_lib.ergodic_metric import ErgodicMetric
            except: pass 

            basis = BasisFunc(n_basis=[8,8], emap=lambda x: jnp.array([
                (x[0]-xb[0])/(xb[1]-xb[0]), (x[1]-yb[0])/(yb[1]-yb[0])]))
            phik = get_phik(distribution.evals, basis)
            ck = get_ck(states, basis, tf, dt)
            erg_metric = float(ErgodicMetric(basis)(ck, phik))
            
            # 【修改5】事后过滤：如果 Metric 真的很差 (>0.5) 才丢弃
            # Solver 里的 erg_ub 是硬约束，这里我们用软约束筛选
            if erg_metric > 0.5: 
                # print(f"  [Drop] Metric {erg_metric:.3f} too high")
                return False

            # 保存
            traj_id = f"{dist_id}_traj_{traj_idx:04d}_g{gamma:.6f}"
            start_list = [float(start_pt[0]), float(start_pt[1])]
            end_list = [float(end_pt[0]), float(end_pt[1])]
            
            with open(os.path.join(self.output_dir, "trajectories", f"{traj_id}.json"), "w") as f:
                json.dump({
                    "id": traj_id,
                    "gamma": float(gamma),
                    "states": states,
                    "controls": controls,
                    "ergodic_metric": erg_metric,
                    "start_point": start_list,
                    "end_point": end_list,
                    "total_time": tf,
                    "solver_converged": sol.get('success', False) if isinstance(sol, dict) else False
                }, f, indent=2, cls=NumpyEncoder)
            
            self.dataset_index["trajectories"].append(traj_id)

            if visualize:
                self._visualize(distribution, states, start_list, end_list, erg_metric, traj_id)
                
            return True
            
        except Exception as e:
            # print(f"Error: {e}")
            import traceback
            traceback.print_exc()
            return False

    def _visualize(self, dist, states, sp, ep, metric, tid):
        try:
            fig, ax = plt.subplots(figsize=(8,6))
            X, Y = dist.domain
            Z = dist.evals[0].reshape(X.shape)
            ax.contourf(X, Y, Z, levels=20, cmap='viridis', alpha=0.6)
            
            states = np.array(states)
            ax.plot(states[:,0], states[:,1], 'r-', lw=2)
            ax.plot(sp[0], sp[1], 'go', ms=10, label='S')
            ax.plot(ep[0], ep[1], 'bo', ms=10, label='E')
            ax.set_title(f"Metric: {metric:.4f}")
            plt.savefig(os.path.join(self.output_dir, "visualizations", f"{tid}.png"))
            plt.close(fig)
        except: pass

if __name__ == "__main__":
    # 配置
    OUTPUT_DIR = "data/ergodic_diagonal"
    
    # 开始采集
    collector = DataCollector(output_dir=OUTPUT_DIR)
    collector.generate_dataset(
        num_distributions=500,     # 先跑5个分布
        trajectories_per_distribution=40,
        visualize=True
    )

#### v2

In [9]:
# data_collection_diagonal_v3_ready.py
import os
import json
import time
import datetime
from tqdm import tqdm
import numpy as np
import jax.numpy as jnp
from jax import vmap
import matplotlib
matplotlib.use('Agg') 
import matplotlib.pyplot as plt

class NumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, jnp.ndarray): return np.array(obj).tolist()
        elif isinstance(obj, (np.ndarray, np.number)): return obj.tolist()
        return super(NumpyEncoder, self).default(obj)
    
class DataCollector:
    def __init__(self, output_dir, workspace_bounds=None):
        self.output_dir = output_dir
        os.makedirs(output_dir, exist_ok=True)
        os.makedirs(os.path.join(output_dir, "distributions"), exist_ok=True)
        os.makedirs(os.path.join(output_dir, "trajectories"), exist_ok=True)
        os.makedirs(os.path.join(output_dir, "visualizations"), exist_ok=True)
        self.workspace_bounds = np.array(workspace_bounds) if workspace_bounds else np.array([[0., 3.5], [-1., 3.5]])
        self.dataset_index = {"created_at": datetime.datetime.now().isoformat(), "distributions": [], "trajectories": []}

    def generate_dataset(self, num_distributions=10, trajectories_per_distribution=5, gamma_range=(0.03, 0.15), visualize=False):
        print(f"🚀 开始采集 (JAX Fixed): {num_distributions} Dists x {trajectories_per_distribution} Trajs")
        gamma_values = np.exp(np.linspace(np.log(gamma_range[0]), np.log(gamma_range[1]), trajectories_per_distribution))
        success_count = 0
        
        for dist_idx in tqdm(range(num_distributions), desc="Generating"):
            dist_seed = int(time.time() * 1000) % 10000 + dist_idx
            distribution = RandomTargetDistribution(self.workspace_bounds, seed=dist_seed)
            
            dist_id = f"dist_{dist_idx:04d}"
            with open(os.path.join(self.output_dir, "distributions", f"{dist_id}.json"), "w") as f:
                json.dump({"id": dist_id, "params": distribution.get_distribution_params()}, f, indent=2, cls=NumpyEncoder)
            self.dataset_index["distributions"].append(dist_id)

            dist_id = f"dist_{dist_idx:04d}"
            with open(os.path.join(self.output_dir, "distributions", f"{dist_id}.json"), "w") as f:
                json.dump({
                    "id": dist_id, 
                    "params": distribution.get_distribution_params(),
                    # 【新增这一行】必须保存边界，否则 Dataset 读取时不知道地图多大
                    "workspace_bounds": self.workspace_bounds.tolist() 
                }, f, indent=2, cls=NumpyEncoder)

            if visualize:
                try: distribution.visualize(save_path=os.path.join(self.output_dir, "visualizations", f"{dist_id}_map.png"))
                except: pass

            for traj_idx, gamma in enumerate(gamma_values):
                if self._generate_diagonal_trajectory(distribution, dist_id, traj_idx, gamma, visualize):
                    success_count += 1
        
        print(f"✅ 完成！成功采集 {success_count} 条轨迹。")
        return self.dataset_index
    
    def _generate_diagonal_trajectory(self, distribution, dist_id, traj_idx, gamma, visualize):
        try:
            xb, yb = self.workspace_bounds[0], self.workspace_bounds[1]
            start_pt = np.array([xb[0] + 0.2, yb[1] - 0.2]) 
            end_pt   = np.array([xb[1] - 0.2, yb[0] + 0.2])
            x0 = np.array([start_pt[0], start_pt[1], 0., 0.])
            xf = np.array([end_pt[0], end_pt[1], 0., 0.])
            dist_val = np.linalg.norm(end_pt - start_pt) 
            tf_estimate = dist_val * 5.0 

            args = {'N': 100, 'x0': x0, 'xf': xf, 'erg_ub': gamma, 'alpha': 0.8, 'wrksp_bnds': self.workspace_bounds}
            
            solver, _ = build_erg_time_opt_solver(args, distribution)
            sol = solver.solve(max_iter=3000, eps=1e-2)
            
            states = None
            if isinstance(sol, dict): states = sol.get('x'); controls = sol.get('u'); tf = float(sol.get('tf', tf_estimate))
            elif hasattr(solver, 'solution') and solver.solution: states = solver.solution.get('x'); controls = solver.solution.get('u'); tf = float(solver.solution.get('tf', tf_estimate))
            
            if states is None or np.isnan(states).any(): return False
            
            # Metric Check
            try:
                from time_opt_erg_lib.fourier_utils import BasisFunc, get_phik, get_ck
                from time_opt_erg_lib.ergodic_metric import ErgodicMetric
                basis = BasisFunc(n_basis=[8,8], emap=lambda x: jnp.array([(x[0]-xb[0])/(xb[1]-xb[0]), (x[1]-yb[0])/(yb[1]-yb[0])]))
                phik = get_phik(distribution.evals, basis)
                ck = get_ck(states, basis, tf, tf/args['N'])
                erg_metric = float(ErgodicMetric(basis)(ck, phik))
                if erg_metric > 0.5: return False
            except: erg_metric = 0.0

            traj_id = f"{dist_id}_traj_{traj_idx:04d}_g{gamma:.6f}"
            start_list = [float(start_pt[0]), float(start_pt[1])]
            end_list = [float(end_pt[0]), float(end_pt[1])]
            
            with open(os.path.join(self.output_dir, "trajectories", f"{traj_id}.json"), "w") as f:
                json.dump({
                    "id": traj_id,
                    "distribution_id": dist_id,
                    "gamma": float(gamma),
                    "states": states,
                    "controls": controls,
                    "ergodic_metric": erg_metric,
                    "total_time": tf,
                    "solve_time": 0.0,
                    "time_step": float(tf/args['N']),
                    "dt": float(tf/args['N']),
                    "start_point": start_list,
                    "end_point": end_list
                }, f, indent=2, cls=NumpyEncoder)
            
            self.dataset_index["trajectories"].append(traj_id)
            if visualize:
                try:
                    fig, ax = plt.subplots(figsize=(8,6))
                    X, Y = distribution.domain; Z = np.array(distribution.evals[0]).reshape(X.shape)
                    ax.contourf(X, Y, Z, levels=20, cmap='viridis', alpha=0.6)
                    s_arr = np.array(states)
                    ax.plot(s_arr[:,0], s_arr[:,1], 'r-', lw=2)
                    ax.plot(start_pt[0], start_pt[1], 'go'); ax.plot(end_pt[0], end_pt[1], 'bo')
                    plt.savefig(os.path.join(self.output_dir, "visualizations", f"{traj_id}.png")); plt.close(fig)
                except: pass
            return True
        except Exception: return False

if __name__ == "__main__":
    # 配置
    OUTPUT_DIR = "data/ergodic_diagonal"
    collector = DataCollector(output_dir=OUTPUT_DIR)
    collector.generate_dataset(num_distributions=500, trajectories_per_distribution=40, visualize=False)

🚀 开始采集 (JAX Fixed): 500 Dists x 40 Trajs


Generating:   0%|          | 0/500 [00:00<?, ?it/s]

done in  1431  iterations 0.009696007
done in  1420  iterations 0.009606361
done in  1383  iterations 0.009890556
done in  1365  iterations 0.009770393
done in  1348  iterations 0.00963974
done in  1311  iterations 0.009924889
done in  1284  iterations 0.009963036
done in  1257  iterations 0.009719372
done in  1226  iterations 0.009768009
done in  1178  iterations 0.00978899
done in  1171  iterations 0.009687424
done in  1154  iterations 0.00927496
done in  1126  iterations 0.0096411705
done in  1089  iterations 0.009781122
done in  1081  iterations 0.009757519
done in  1062  iterations 0.009404421
done in  1047  iterations 0.009746313
done in  1035  iterations 0.009140611
done in  1017  iterations 0.009566426
done in  1000  iterations 0.008818954
done in  945  iterations 0.009360075
done in  934  iterations 0.008840889
done in  911  iterations 0.009814888
done in  898  iterations 0.009213835
done in  868  iterations 0.0085712075
done in  839  iterations 0.008778334
done in  804  itera

Generating:   0%|          | 1/500 [00:49<6:50:06, 49.31s/it]

done in  538  iterations 0.009221554
done in  1053  iterations 0.009063721
done in  1071  iterations 0.009643555
done in  1066  iterations 0.009811401
done in  1441  iterations 0.009613037
done in  1072  iterations 0.009780884
done in  1055  iterations 0.008430481
done in  1057  iterations 0.009048462
done in  1047  iterations 0.00995636
done in  1042  iterations 0.009140015
done in  1033  iterations 0.009712219
done in  1041  iterations 0.009437561
done in  1042  iterations 0.009727478
done in  1050  iterations 0.0090789795
done in  1038  iterations 0.00920105
done in  1032  iterations 0.0079956055
done in  1038  iterations 0.009689331
done in  1051  iterations 0.008926392
done in  1047  iterations 0.00945282
done in  1030  iterations 0.008583069
done in  1042  iterations 0.009544373
done in  1032  iterations 0.009971619
done in  1045  iterations 0.009315491
done in  2092  iterations 0.008956909
done in  1422  iterations 0.0098724365
done in  1051  iterations 0.00945282
done in  1061 

...

done in  91  iterations 0.009393692
done in  1614  iterations 0.009908676
done in  1444  iterations 0.00976944
done in  1455  iterations 0.009984016
done in  1456  iterations 0.0099487305
done in  1413  iterations 0.009583473
done in  1386  iterations 0.009387016
done in  1379  iterations 0.009626389
done in  1349  iterations 0.009493828
done in  1330  iterations 0.009879589
done in  1303  iterations 0.009856701
done in  1214  iterations 0.00974226
done in  1193  iterations 0.00952816
done in  1189  iterations 0.009806156
done in  1172  iterations 0.009665489
done in  1112  iterations 0.009436607
done in  1090  iterations 0.009667397
done in  1074  iterations 0.00896889
done in  1049  iterations 0.009255499
done in  1022  iterations 0.009702921
done in  954  iterations 0.009753704
done in  961  iterations 0.009224653
done in  80  iterations 0.0028381348
done in  951  iterations 0.009877682
done in  917  iterations 0.009771824
done in  909  iterations 0.009223938
done in  880  iteration

Generating: 100%|█████████▉| 499/500 [8:09:39<00:46, 46.93s/it]

done in  78  iterations 0.008773804
done in  1959  iterations 0.009429932
done in  1957  iterations 0.009662628
done in  1952  iterations 0.009672165
done in  1918  iterations 0.009870529
done in  1869  iterations 0.009849548
done in  1828  iterations 0.009939194
done in  1753  iterations 0.00996685
done in  1699  iterations 0.009695053
done in  1666  iterations 0.009742737
done in  1637  iterations 0.009731293
done in  1608  iterations 0.009681702
done in  1569  iterations 0.009846687
done in  1520  iterations 0.009908676
done in  1477  iterations 0.009628296
done in  1440  iterations 0.009635448
done in  1413  iterations 0.009601116
done in  1383  iterations 0.00956583
done in  1355  iterations 0.009763956
done in  1327  iterations 0.009863377
done in  1299  iterations 0.009819865
done in  1272  iterations 0.009932891
done in  1250  iterations 0.00895977
done in  1229  iterations 0.00918293
done in  1205  iterations 0.009905815
done in  1179  iterations 0.00905776
done in  1150  iter

Generating: 100%|██████████| 500/500 [8:10:39<00:00, 58.88s/it]

done in  862  iterations 0.00880909
✅ 完成！成功采集 19996 条轨迹。


### 数据格式检验

In [8]:
# cell_test_dataset.py
import torch
from torch.utils.data import DataLoader
# 假设上面的 ErgodicDataset 类定义在 diffusion_ergodic/data_process/ergodic_processor.py
# 如果不是，请调整 import，或者直接把上面的类定义粘贴到这里
# Cell 02: Data module (v3.5: Gamma Injection + Safe Gaussian Params)

import os, json, numpy as np, torch
from torch.utils.data import Dataset, DataLoader, SubsetRandomSampler

def as_tensor_f32(x):
    return x.to(torch.float32) if isinstance(x, torch.Tensor) else torch.as_tensor(x, dtype=torch.float32)

class ErgodicDataset(Dataset):
    def __init__(self, data_dir, transform=None, max_trajectory_len=101, use_index=True, max_gaussians=20):
        self.data_dir = data_dir
        self.transform = transform
        self.max_trajectory_len = max_trajectory_len
        self.max_gaussians = max_gaussians
        
        self.distributions_dir = os.path.join(data_dir, 'distributions')
        self.trajectories_dir  = os.path.join(data_dir, 'trajectories')
        self.data_pairs = []

        index_path = os.path.join(data_dir, 'dataset_index.json')
        
        index_loaded = False
        if use_index and os.path.exists(index_path):
            try:
                with open(index_path, 'r') as f:
                    loaded_data = json.load(f)
                if isinstance(loaded_data, list):
                    self.data_pairs = loaded_data
                    index_loaded = True
            except: pass
        
        if not index_loaded or len(self.data_pairs) == 0:
            self._build_index(index_path)
            
    def _build_index(self, index_path):
        self.data_pairs = [] 
        dist_files = [f for f in os.listdir(self.distributions_dir) if f.endswith('.json')]
        dist_id_to_file = {d: d for d in dist_files} # Simplified map
        # Build accurate map
        dist_id_to_file = {}
        for df in dist_files:
            try:
                with open(os.path.join(self.distributions_dir, df), 'r') as f:
                    d = json.load(f)
                dist_id_to_file[d['id']] = df
            except: pass

        for tf in os.listdir(self.trajectories_dir):
            if tf.endswith('.json'):
                try:
                    with open(os.path.join(self.trajectories_dir, tf), 'r') as f:
                        t = json.load(f)
                    did = t['distribution_id']
                    if did in dist_id_to_file:
                        self.data_pairs.append({
                            'distribution_file': dist_id_to_file[did],
                            'trajectory_file': tf
                        })
                except: pass
        try:
            with open(index_path, 'w') as f:
                json.dump(self.data_pairs, f)
        except: pass

    def __len__(self):
        return len(self.data_pairs)

    def _generate_distribution_grid(self, dist_data, grid_size=(32,32)):
        bounds = dist_data.get('bounds', [[0,3],[0,3]])
        centers = np.asarray(dist_data['params']['centers'])
        covs    = np.asarray(dist_data['params']['covs'])
        weights = np.asarray(dist_data['params']['weights'])
        n = int(dist_data['params']['n_gaussians'])
        x = np.linspace(bounds[0][0], bounds[0][1], grid_size[0])
        y = np.linspace(bounds[1][0], bounds[1][1], grid_size[1])
        X, Y = np.meshgrid(x, y)
        Z = np.zeros_like(X, dtype=np.float64)
        for i in range(n):
            cx, cy = centers[i]
            c = covs[i]
            if np.isscalar(c): sx, sy = c, c
            elif len(np.shape(c)) == 1: sx, sy = c[0], c[1]
            else: sx, sy = np.sqrt(np.diag(c))
            Z += weights[i] * np.exp(-(((X-cx)**2)/(2*sx**2 + 1e-8) + ((Y-cy)**2)/(2*sy**2 + 1e-8)))
        Z /= (Z.max() + 1e-8)
        return Z

    def _process_trajectory(self, traj_data):
        states = np.asarray(traj_data['states'], dtype=np.float64)
        pos = states[:, :2]
        vel = np.zeros((states.shape[0],1))
        if states.shape[1] > 3: vel = states[:, 3:4]
        elif states.shape[0] > 1: 
            dt = float(traj_data['time_step'])
            d = np.linalg.norm(np.diff(pos, axis=0), axis=1)
            vel = np.vstack([np.zeros((1,1)), (d/dt)[:,None]])
        heading = states[:, 2:3] if states.shape[1] > 2 else np.zeros((states.shape[0],1))
        rs = np.hstack([pos, heading, vel])
        
        T = self.max_trajectory_len
        controls = np.zeros((T, 2), dtype=np.float32) # 强制全零
        
        if len(rs) > T:
            idx = np.linspace(0, len(rs)-1, T).astype(int)
            rs = rs[idx]
        elif len(rs) < T:
            pad = T - len(rs)
            rs = np.vstack([rs, np.zeros((pad, rs.shape[1]))])
            
        return rs, controls, float(traj_data['time_step']), float(traj_data['total_time']), float(traj_data['ergodic_metric']), float(traj_data['gamma'])

    def _pack_gaussian_params(self, dist_data):
        # (保持之前的实现)
        n = int(dist_data['params']['n_gaussians'])
        centers = np.asarray(dist_data['params']['centers'])
        weights = np.asarray(dist_data['params']['weights'])
        covs_raw = dist_data['params']['covs']
        packed = np.zeros((self.max_gaussians, 7), dtype=np.float32)
        mask = np.zeros((self.max_gaussians), dtype=bool)
        mask[n:] = True
        for i in range(min(n, self.max_gaussians)):
            packed[i, 0:2] = centers[i]
            c = covs_raw[i]
            if np.isscalar(c): cov_mat = np.array([c, 0, 0, c])
            elif len(np.shape(c)) == 1: cov_mat = np.array([c[0], 0, 0, c[1]])
            else: cov_mat = np.asarray(c).flatten()
            if cov_mat.size == 4: packed[i, 2:6] = cov_mat
            packed[i, 6] = weights[i]
        return packed, mask

    def __getitem__(self, idx):
        pair = self.data_pairs[idx]
        with open(os.path.join(self.distributions_dir, pair['distribution_file']), 'r') as f:
            dist_data = json.load(f)
        with open(os.path.join(self.trajectories_dir, pair['trajectory_file']), 'r') as f:
            traj_data = json.load(f)

        distribution_grid = self._generate_distribution_grid(dist_data) 
        rs, controls, time_step, total_time, ergodic_metric, gamma = self._process_trajectory(traj_data)
        gmm_packed, gmm_padding_mask = self._pack_gaussian_params(dist_data)
        
        # --- 核心修复：使用 Padded 数据填充 gaussian_params ---
        # 这样即使原始数据变长，传给 DataLoader 的也是定长 Tensor
        centers_padded = as_tensor_f32(gmm_packed[:, 0:2]) 
        covs_padded    = as_tensor_f32(gmm_packed[:, 2:6])
        weights_padded = as_tensor_f32(gmm_packed[:, 6])
        n_gauss = int(dist_data['params']['n_gaussians'])
        
        sample = {
            "distribution": as_tensor_f32(distribution_grid).unsqueeze(0), 
            "robot_state": as_tensor_f32(rs[0]),
            "trajectories": as_tensor_f32(rs),
            "controls": as_tensor_f32(controls),
            "gamma": as_tensor_f32(gamma).view(1), # Gamma 注入
            
            "gaussian_packed": as_tensor_f32(gmm_packed),        
            "gaussian_padding_mask": torch.tensor(gmm_padding_mask, dtype=torch.bool), 
            
            # 这里的修改消除了 [5,2] vs [7,2] 的冲突
            "gaussian_params": {
                "n_gaussians": torch.tensor(n_gauss, dtype=torch.long), 
                "centers": centers_padded, # [20, 2]
                "covs": covs_padded,       # [20, 4]
                "weights": weights_padded  # [20]
            }
        }
        return sample

def build_loaders(data_dir, batch_size=64, max_trajectory_len=101, val_split=0.1, num_workers=0, seed=42, transform=None):
    ds = ErgodicDataset(data_dir=data_dir, transform=transform, max_trajectory_len=max_trajectory_len)
    N = len(ds); idx = list(range(N))
    split = int(np.floor(val_split * N))
    rng = np.random.RandomState(seed); rng.shuffle(idx)
    val_idx, train_idx = idx[:split], idx[split:]
    train_loader = DataLoader(ds, batch_size=batch_size, sampler=SubsetRandomSampler(train_idx),
                              num_workers=num_workers, pin_memory=True, drop_last=False)
    val_loader   = DataLoader(ds, batch_size=batch_size, sampler=SubsetRandomSampler(val_idx),
                              num_workers=num_workers, pin_memory=True, drop_last=False)
    return train_loader, val_loader

print("ErgodicDataset ready (v3.5: Gamma + Safe Gaussian Params).")

# 配置路径
DATA_DIR = "data/ergodic_diagonal_v3_ready" # 指向你刚采集好的目录

print("⚡️ 正在进行最终格式验证...")

# 1. 初始化
try:
    # 注意：这里不需要 transform，我们只测读取
    ds = ErgodicDataset(data_dir=DATA_DIR, max_trajectory_len=100)
    print(f"✅ Dataset 初始化成功，共发现 {len(ds)} 条数据。")
except Exception as e:
    print(f"❌ 初始化失败: {e}")
    exit()

# 2. 模拟训练读取 (Batch Test)
# batch_size=4 只要能跑通，batch_size=256 也能跑通
loader = DataLoader(ds, batch_size=4, shuffle=True)

try:
    for batch in loader:
        # 检查关键数据流
        print("-" * 30)
        print("📦 成功提取一个 Batch:")
        print(f"   Shape [Batch, Len, Dim]")
        print(f"   - Distribution Map: {batch['distribution'].shape} (预期: [4, 1, 32, 32])")
        print(f"   - Robot State:      {batch['robot_state'].shape}    (预期: [4, 4])")
        print(f"   - Gamma:            {batch['gamma'].shape}          (预期: [4, 1])")
        print(f"   - Gaussian Centers: {batch['gaussian_params']['centers'].shape} (预期: [4, 20, 2])")
        
        # 检查是否有 NaN
        if torch.isnan(batch['trajectories']).any():
            print("❌ 警告：轨迹数据包含 NaN！")
        else:
            print("✅ 数据数值正常 (无 NaN)")
            
        break # 只测一个 Batch 就够了
        
    print("-" * 30)
    print("🎉 恭喜！数据格式完美，可以直接开始训练！")

except Exception as e:
    print(f"❌ DataLoader 读取失败: {e}")
    print("原因可能是: 某个字段维度不统一，导致无法 Stack 成 Batch。")

ErgodicDataset ready (v3.5: Gamma + Safe Gaussian Params).
⚡️ 正在进行最终格式验证...
✅ Dataset 初始化成功，共发现 20 条数据。
------------------------------
📦 成功提取一个 Batch:
   Shape [Batch, Len, Dim]
   - Distribution Map: torch.Size([4, 1, 32, 32]) (预期: [4, 1, 32, 32])
   - Robot State:      torch.Size([4, 4])    (预期: [4, 4])
   - Gamma:            torch.Size([4, 1])          (预期: [4, 1])
   - Gaussian Centers: torch.Size([4, 20, 2]) (预期: [4, 20, 2])
✅ 数据数值正常 (无 NaN)
------------------------------
🎉 恭喜！数据格式完美，可以直接开始训练！


## Cell: Data Verification & Analytics (Full Version)

In [ ]:
# ## Cell: Data Verification & Analytics (Full Version)
import os
import json
import glob
import random
import numpy as np
import matplotlib.pyplot as plt
# 【关键修复】使用标准 tqdm，避免 ipywidgets 报错
from tqdm import tqdm 

# 配置：数据路径
DATASET_DIR = "/home/songxy/code/Diffusion-Ergodic/diffusion_ergodic/data/ergodic_dataset_no_obs"
TRAJ_DIR = os.path.join(DATASET_DIR, "trajectories")
DIST_DIR = os.path.join(DATASET_DIR, "distributions")

# ==========================================
# 1. 辅助函数：计算轨迹的"弯曲度" (Linearity)
# ==========================================
def analyze_trajectory_geometry(states):
    """
    计算轨迹几何特征
    Linearity Ratio = 直线距离 / 实际路径长度
    - 接近 1.0 : 直线 (Bad for us)
    - 接近 0.5 : 强烈的 S 型或 U 型 (Good for us)
    """
    states = np.array(states)
    pos = states[:, :2]
    
    # 1. 欧氏距离 (起点到终点)
    start = pos[0]
    end = pos[-1]
    euclidean_dist = np.linalg.norm(end - start)
    
    # 2. 实际路径长度 (累加每一步)
    steps = np.linalg.norm(pos[1:] - pos[:-1], axis=1)
    path_length = np.sum(steps)
    
    # 3. 线性度 (越小越弯)
    if path_length < 1e-6: return 1.0 # 没动
    linearity = euclidean_dist / path_length
    
    return linearity, path_length, euclidean_dist

# ==========================================
# 2. 随机抽样可视化 (The Eye Test)
# ==========================================
def visualize_random_samples(num_samples=9):
    json_files = glob.glob(os.path.join(TRAJ_DIR, "*.json"))
    if not json_files:
        print("❌ 没有找到任何轨迹文件！请检查路径。")
        return

    # 随机采样
    samples = random.sample(json_files, min(num_samples, len(json_files)))
    
    # 创建画布
    fig, axes = plt.subplots(3, 3, figsize=(15, 15))
    axes = axes.flatten()
    
    print(f"👁️ 正在可视化 {len(samples)} 个随机样本...")
    
    for i, traj_file in enumerate(samples):
        try:
            with open(traj_file, 'r') as f:
                data = json.load(f)
                
            # 加载对应的分布文件来画背景
            dist_id = data['distribution_id']
            dist_file = os.path.join(DIST_DIR, f"{dist_id}.json")
            
            ax = axes[i]
            
            # 1. 画分布 (如果有的话)
            if os.path.exists(dist_file):
                with open(dist_file, 'r') as f:
                    dist_data = json.load(f)
                    params = dist_data['params']
                    # 简易画法：只画中心点
                    centers = np.array(params['centers'])
                    ax.scatter(centers[:,0], centers[:,1], c='red', s=150, alpha=0.3, label='Hotspots')
            
            # 2. 画轨迹
            states = np.array(data['states'])
            linearity, length, _ = analyze_trajectory_geometry(states)
            
            # 根据弯曲度变色：越弯越紫(Magenta)，越直越青(Cyan)
            color = 'magenta' if linearity < 0.8 else 'cyan'
            
            ax.plot(states[:,0], states[:,1], color=color, linewidth=2.5, label='Trajectory')
            ax.scatter(states[0,0], states[0,1], c='green', marker='o', s=50, label='Start')
            ax.scatter(states[-1,0], states[-1,1], c='blue', marker='x', s=50, label='End')
            
            metric_val = data.get('ergodic_metric', 0.0)
            ax.set_title(f"ID: {dist_id}\nLinearity: {linearity:.2f} (Low is Good)\nMetric: {metric_val:.4f}")
            ax.set_xlim(0, 3.5); ax.set_ylim(-1, 3.5)
            
            # 只在第一个图显示图例
            if i == 0: ax.legend(loc='upper right')
            
        except Exception as e:
            print(f"Error plotting {traj_file}: {e}")

    plt.tight_layout()
    plt.show()

# ==========================================
# 3. 统计分析 (The Math Test)
# ==========================================
def run_statistical_analysis(max_files=1000):
    json_files = glob.glob(os.path.join(TRAJ_DIR, "*.json"))
    total_files = len(json_files)
    
    if total_files == 0: return
    
    print(f"📊 正在分析前 {min(max_files, total_files)} / {total_files} 条数据...")
    
    linearities = []
    metrics = []
    
    # 随机打乱以获得无偏统计
    random.shuffle(json_files)
    
    # 使用标准 tqdm，指定 ncols 防止格式错乱
    for fpath in tqdm(json_files[:max_files], ncols=80):
        try:
            with open(fpath, 'r') as f:
                data = json.load(f)
            
            states = data['states']
            lin, _, _ = analyze_trajectory_geometry(states)
            
            linearities.append(lin)
            metrics.append(data.get('ergodic_metric', 0))
            
        except Exception as e:
            continue
            
    # 绘图
    fig, ax = plt.subplots(1, 3, figsize=(18, 5))
    
    # 直方图 1: 弯曲度 (Linearity)
    ax[0].hist(linearities, bins=30, color='purple', alpha=0.7, edgecolor='black')
    ax[0].axvline(0.9, color='red', linestyle='--', label='Straight Limit')
    ax[0].set_title("Linearity Distribution\n(Left is Better/Curvier)")
    ax[0].set_xlabel("Linearity Ratio")
    ax[0].legend()
    
    # 直方图 2: Ergodic Metric
    ax[1].hist(metrics, bins=30, color='orange', alpha=0.7, edgecolor='black')
    ax[1].set_title("Ergodic Metric Distribution\n(Lower is Better)")
    ax[1].set_xlabel("Metric Value")
    
    # 散点图 3: 弯曲度 vs Metric
    ax[2].scatter(linearities, metrics, alpha=0.3, c='teal')
    ax[2].set_xlabel("Linearity (Straightness)")
    ax[2].set_ylabel("Ergodic Metric")
    ax[2].set_title("Correlation: Curvature vs Quality")
    ax[2].invert_xaxis() # 让左边（弯曲）在左边
    
    plt.tight_layout()
    plt.show()
    
    # 打印核心指标
    avg_lin = np.mean(linearities)
    print(f"\n🏆 数据集质量总结:")
    print(f"   - 平均线性度 (Linearity): {avg_lin:.4f} (越低越好)")
    print(f"   - 弯曲轨迹占比 (< 0.8):  {np.mean(np.array(linearities) < 0.8):.1%}")
    print(f"   - 平均 Ergodic Metric:   {np.mean(metrics):.4f}")

# --- 执行 ---
visualize_random_samples()
run_statistical_analysis()

### Cell: Verify OLD Dataset (Contrast Analysis)

In [ ]:
# ## Cell: Verify OLD Dataset (Contrast Analysis)
import os
# 不需要重新 import 函数，直接复用内存里的

# 1. 指向旧数据路径
# 请确认这是你之前生成的、带有避障约束的旧数据集路径
DATASET_DIR = "/home/songxy/code/Diffusion-Ergodic/diffusion_ergodic/ergodic_dataset_wild_full" 

# 2. 更新子路径
TRAJ_DIR = os.path.join(DATASET_DIR, "trajectories")
DIST_DIR = os.path.join(DATASET_DIR, "distributions")

print(f"📉 正在分析旧数据集: {DATASET_DIR}")

# 3. 执行同样的分析
# 我预测这里出来的图全是青色的直线，直方图会堆积在 1.0 附近
visualize_random_samples()
run_statistical_analysis()